In [1]:

import sqlite3
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

In [2]:
# 1) Ler dados
# ================================

db_path = r"C:\Users\beasa\Desktop\AASE\ScreenTimevsMentalWellness.db"
table_main = "ScreenTimevsMentalWellness"

conn = sqlite3.connect(db_path)
df = pd.read_sql_query(f"SELECT * FROM {table_main}", conn)
conn.close()

print("Dimensão inicial:", df.shape)

target = "mental_wellness_index_0_100"

if target not in df.columns:
    raise Exception("Coluna alvo não encontrada.")

y = df[target]
mask = y.notna()
df = df[mask]
y = y[mask]

Dimensão inicial: (399, 20)


In [3]:

# 2) Features do Cenário C
#    Apenas variáveis comportamentais
# ================================

features_C = [
    "screen_time_hours",
    "work_screen_hours",
    "leisure_screen_hours",
    "sleep_hours",
    "sleep_quality_1_5",
    "exercise_minutes_per_week",
    "social_hours_per_week",
    "stress_level_0_10",
    "productivity_0_100"
]

features_C = [c for c in features_C if c in df.columns]
X = df[features_C].fillna(df[features_C].mean())

print("Features usadas no Cenário C:", features_C)
print("Dimensão do X:", X.shape)

Features usadas no Cenário C: ['screen_time_hours', 'work_screen_hours', 'leisure_screen_hours', 'sleep_hours', 'sleep_quality_1_5', 'exercise_minutes_per_week', 'social_hours_per_week', 'stress_level_0_10', 'productivity_0_100']
Dimensão do X: (399, 9)


In [4]:
# 3) Modelos – 5 técnicas
# ================================

scaler = StandardScaler()

models = {
    "LinearRegression": Pipeline([
        ("scaler", scaler),
        ("model", LinearRegression())
    ]),

    "RandomForest": Pipeline([
        ("scaler", scaler),
        ("model", RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "DecisionTree": Pipeline([
        ("scaler", scaler),
        ("model", DecisionTreeRegressor(random_state=42))
    ]),

    "AdaBoost": Pipeline([
        ("scaler", scaler),
        ("model", AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=4),
            n_estimators=200,
            random_state=42
        ))
    ]),

    "GradientBoosting": Pipeline([
        ("scaler", scaler),
        ("model", GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            random_state=42
        ))
    ])
}

In [5]:
# 4) Cross-Validation 10-fold
# ================================

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

print("\n===== CENÁRIO C — 5 Técnicas de Regressão (apenas variáveis comportamentais) =====\n")

results = []

for name, pipe in models.items():
    print(f"A avaliar modelo: {name}...")
    
    cv = cross_validate(
        pipe, X, y, cv=kfold,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2"
        }
    )

    mae = -cv["test_MAE"].mean()
    rmse = -cv["test_RMSE"].mean()
    r2 = cv["test_R2"].mean()

    print(f"Modelo: {name}")
    print(f" MAE:  {mae:.3f}")
    print(f" RMSE: {rmse:.3f}")
    print(f" R2:   {r2:.3f}\n")

    results.append({
        "Cenário": "C",
        "Modelo": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

df_results = pd.DataFrame(results)



===== CENÁRIO C — 5 Técnicas de Regressão (apenas variáveis comportamentais) =====

A avaliar modelo: LinearRegression...
Modelo: LinearRegression
 MAE:  4.093
 RMSE: 5.191
 R2:   0.931

A avaliar modelo: RandomForest...
Modelo: RandomForest
 MAE:  4.963
 RMSE: 6.327
 R2:   0.898

A avaliar modelo: DecisionTree...
Modelo: DecisionTree
 MAE:  7.161
 RMSE: 9.528
 R2:   0.763

A avaliar modelo: AdaBoost...
Modelo: AdaBoost
 MAE:  5.331
 RMSE: 6.568
 R2:   0.889

A avaliar modelo: GradientBoosting...
Modelo: GradientBoosting
 MAE:  4.728
 RMSE: 6.062
 R2:   0.906



In [6]:
# 5) Tabela final bonita
# ================================

df_results["MAE"] = df_results["MAE"].round(3)
df_results["RMSE"] = df_results["RMSE"].round(3)
df_results["R2"] = df_results["R2"].round(3)

df_results = df_results.sort_values(by="RMSE")

df_results = df_results.rename(columns={
    "Cenário": "Cenário",
    "Modelo": "Modelo",
    "MAE": "MAE",
    "RMSE": "RMSE",
    "R2": "R²"
})

print("\nResultados finais Cenário C:\n")
print(df_results.to_string(index=False, justify="center"))


Resultados finais Cenário C:

Cenário      Modelo        MAE  RMSE   R² 
   C    LinearRegression 4.093 5.191 0.931
   C    GradientBoosting 4.728 6.062 0.906
   C        RandomForest 4.963 6.327 0.898
   C            AdaBoost 5.331 6.568 0.889
   C        DecisionTree 7.161 9.528 0.763
